### Fase 4: Pré-processamento (Compliance).

Esta etapa é crucial para garantir a pontuação técnica no desafio. Embora modelos Transformer modernos (como o RoBERTa que usamos no Teacher) prefiram texto bruto, o edital exige técnicas clássicas de NLP (Remoção de Stopwords, Lematização, etc.).

Para o Modelo Student (que será mais leve, provavelmente um DistilBERT ou até uma Regressão Logística para baseline), esse texto limpo ajuda a focar nas palavras-chave essenciais.

📋 O Plano de Execução
Ferramenta: Usaremos spaCy. Ele é industrial, muito mais rápido que o NLTK para lematização e permite processamento em lote (pipe).

Estratégia de "Smart Stopwords": Removeremos as stopwords padrão do inglês, EXCETO as negações (no, not, never, nor).

Por que? Se removermos "not" de "not good", vira "good" (sentimento oposto). Manter a negação preserva a semântica.

Lematização: Converteremos verbos e plurais para a raiz (ex: "complaining" -> "complain", "banks" -> "bank").

Limpeza: Remover pontuação e caracteres especiais, mantendo apenas letras.

[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [1]:
import polars as pl
import spacy
import os
from pathlib import Path
from tqdm import tqdm

# --- Configurações e Caminhos ---
BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / "data" / "parquet" / "training_dataset_2025.parquet"
OUTPUT_FILE = BASE_DIR / "data" / "parquet" / "processed_dataset.parquet"

# Lista de palavras para NÃO remover (Negações são vitais para sentimento)
KEEP_WORDS = {'no', 'not', 'nor', 'neither', 'never', 'none', 'but', 'however', 'although'}

def load_spacy_model():
    print("⏳ Carregando modelo spaCy (en_core_web_sm)...")
    try:
        # Desabilitamos 'parser' e 'ner' pois só precisamos de tokenização e lematização
        # Isso acelera o processo em 10x
        nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
        return nlp
    except OSError:
        print("❌ Modelo não encontrado. Execute: python -m spacy download en_core_web_sm")
        raise

def get_optimized_stopwords(nlp):
    """
    Retorna o set de stopwords do spaCy MENOS as palavras de negação.
    """
    default_stops = nlp.Defaults.stop_words
    # Removemos as palavras que queremos manter da lista de exclusão
    final_stops = default_stops - KEEP_WORDS
    print(f"ℹ️ Stopwords configuradas. Mantendo negações: {list(KEEP_WORDS)}")
    return final_stops

def preprocess_compliance_phase():
    print(f"--- 🧹 FASE 4: Pré-processamento (Compliance & Cleaning) ---")
    
    if not INPUT_FILE.exists():
        print(f"❌ Arquivo de entrada não encontrado: {INPUT_FILE}")
        return

    # 1. Carregar Dados
    print(f"📂 Lendo Gold Dataset: {INPUT_FILE}")
    df = pl.read_parquet(INPUT_FILE)
    print(f"   Total de registros: {len(df):,}")

    # 2. Configurar NLP
    nlp = load_spacy_model()
    stop_words = get_optimized_stopwords(nlp)

    # 3. Processamento em Batch (spaCy Pipe)
    # Convertemos para lista python para usar o .pipe() que é C-optimized
    texts = df["text"].to_list()
    cleaned_texts = []

    print("🚀 Iniciando limpeza e lematização (Isso pode levar alguns minutos)...")
    
    # n_process=1 é mais seguro no Windows/Jupyter. Se estiver no Linux, pode aumentar.
    # batch_size=2000 é um bom equilíbrio para RAM.
    for doc in tqdm(nlp.pipe(texts, batch_size=2000, n_process=1), total=len(texts), desc="Processing"):
        tokens_limpos = []
        
        for token in doc:
            # Filtros:
            # 1. Não é pontuação
            # 2. Não é espaço em branco
            # 3. Não é stopword (exceto negações)
            # 4. É alfabético (opcional, remove números e sujeira)
            if (not token.is_punct and 
                not token.is_space and 
                token.lower_ not in stop_words and
                token.is_alpha): # Remove números e símbolos estranhos
                
                # Aplica Lematização e Lowercase
                tokens_limpos.append(token.lemma_.lower())
        
        # Reconstrói o texto
        cleaned_texts.append(" ".join(tokens_limpos))

    # 4. Adicionar ao DataFrame e Salvar
    print("💾 Consolidando e salvando...")
    
    # Criamos o DataFrame final com a nova coluna
    df_final = df.with_columns(
        pl.Series(name="cleaned_text", values=cleaned_texts)
    )
    
    # Filtra linhas que ficaram vazias após a limpeza (ex: texto só tinha pontuação)
    df_final = df_final.filter(pl.col("cleaned_text").str.len_chars() > 2)

    df_final.write_parquet(OUTPUT_FILE)
    
    print(f"✅ Sucesso! Arquivo processado salvo em: {OUTPUT_FILE}")
    
    # Comparativo Antes vs Depois
    print("\n--- 🔍 Comparativo (Antes vs Depois) ---")
    amostra = df_final.sample(3)
    for row in amostra.iter_rows(named=True):
        print(f"ORIGINAL: {row['text'][:100]}...")
        print(f"LIMPO:    {row['cleaned_text'][:100]}...")
        print("-" * 20)

if __name__ == "__main__":
    preprocess_compliance_phase()

--- 🧹 FASE 4: Pré-processamento (Compliance & Cleaning) ---
📂 Lendo Gold Dataset: F:\workspace\postech\datathon_tech_challenge_5\data\parquet\training_dataset_2025.parquet
   Total de registros: 484,024
⏳ Carregando modelo spaCy (en_core_web_sm)...
ℹ️ Stopwords configuradas. Mantendo negações: ['but', 'however', 'no', 'although', 'never', 'nor', 'neither', 'none', 'not']
🚀 Iniciando limpeza e lematização (Isso pode levar alguns minutos)...


Processing: 100%|█████████████████████████████████████████████████████████████| 484024/484024 [59:59<00:00, 134.48it/s]


💾 Consolidando e salvando...
✅ Sucesso! Arquivo processado salvo em: F:\workspace\postech\datathon_tech_challenge_5\data\parquet\processed_dataset.parquet

--- 🔍 Comparativo (Antes vs Depois) ---
ORIGINAL: I have a goal of getting a house as soon as possible but the stuff on my credit report will really p...
LIMPO:    goal get house soon possible but stuff credit report trouble ve try understand instance credit burea...
--------------------
ORIGINAL: I am writing to remove the following information from my file. The items I deleted are listed in the...
LIMPO:    write remove follow information file item delete list report not authorization report lack authoriza...
--------------------
ORIGINAL: The personal information on my credit report were outdated....
LIMPO:    personal information credit report outdate...
--------------------



[Voltar para notebook principal](./0_notebook_principal.ipynb)